# Summary Format

In [1]:
# --- Define the Structured Template ---
SUMMARY_TEMPLATE = (
    "The test confirms the [PART_NUMBER] result was [TEST_RESULT]. "
    "Key findings: [FAILURE_MODE] observed at [DURATION]."
)

In [2]:
# --- Create the LLM Prompt ---
def create_structured_prompt(masked_input_text, required_template):
    """
    Constructs a detailed prompt to instruct the LLM on its task
    and the required output structure.
    """
    # System Instruction: Tells the LLM its role
    system_instruction = (
        "You are an expert Aerospace Test Report Summarization AI. "
        "Your goal is to extract key facts and generate a summary "
        "that strictly adheres to the provided output format."
    )
    
    # User Query: Provides the data and the template
    user_query = f"""
    SUMMARY TASK: Extract the key facts from the following test report snippet 
    and output them *only* in the format provided below.
    
    REPORT SNIPPET (Masked Input):
    ---
    {masked_input_text} 
    ---
    
    REQUIRED OUTPUT FORMAT (Fill in the generic tokens):
    {required_template}
    """
    
    return system_instruction + "\n\n" + user_query

In [3]:
# --- Conceptual LLM Call ---
def conceptual_llm_call(prompt):
    # This simulates the fine-tuned Mistral-7B generating the output.
    # It would have learned to fill the slots based on the masked input.
    
    # In a real scenario, you'd call a library like 'transformers'
    # and pass the prompt to the Mistral-7B model.
    
    # We are simulating the expected output:
    simulated_output = SUMMARY_TEMPLATE.replace(
        "[PART_NUMBER]", "AX-430" # The model extracts the specific ID
    ).replace(
        "[TEST_RESULT]", "PASSED"
    ).replace(
        "[FAILURE_MODE]", "no failure observed"
    ).replace(
        "[DURATION]", "100 hours"
    )
    
    return simulated_output

## Working ...

In [4]:
# Get the masked text from the previous step (conceptually)
dummy_masked_text = "The failure was observed on the [PART_NUMBER] from [ORGANIZATION], occurring at [DURATION]."

In [5]:
# Generate the prompt
final_prompt = create_structured_prompt(dummy_masked_text, SUMMARY_TEMPLATE)
print(final_prompt)

You are an expert Aerospace Test Report Summarization AI. Your goal is to extract key facts and generate a summary that strictly adheres to the provided output format.


    SUMMARY TASK: Extract the key facts from the following test report snippet 
    and output them *only* in the format provided below.

    REPORT SNIPPET (Masked Input):
    ---
    The failure was observed on the [PART_NUMBER] from [ORGANIZATION], occurring at [DURATION]. 
    ---

    REQUIRED OUTPUT FORMAT (Fill in the generic tokens):
    The test confirms the [PART_NUMBER] result was [TEST_RESULT]. Key findings: [FAILURE_MODE] observed at [DURATION].
    


In [6]:
# Get the structured summary
structured_summary = conceptual_llm_call(final_prompt)
print("\n--- Model Output ---")
print(structured_summary)


--- Model Output ---
The test confirms the AX-430 result was PASSED. Key findings: no failure observed observed at 100 hours.


## Explaination

The fine-tuned LLM is supposed to produce a summary in a specific, structured format using custom tags, like this:

"The test on part [PART_NUMBER] resulted in a [FAILURE_MODE] after [DURATION] of operation."

Sometimes, though, the LLM gets a little creative and adds extra flavor or misses a key ingredient:

Raw LLM Output: "Okay, here is the summary: The test on part [PART_NUMBER] resulted in a [FAILURE_MODE] after [DURATION] of operation. I hope this helps!"

> This script performs three main jobs on that raw text:
> - Check if all required tags are present in the summary. If any tag is missing, summary is flagged as incomplete.
> - It cleans any unwanted text - example, "I hope this helps!" 
> - Package the order - It takes the clean text, finds the data inside the tags, and converts it into a structured JSON format.